In [1]:
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever
from langchain_classic.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("CH10-Retriever")

doc_list = [
    "I like apples",
    "I like apple company",
    "I like apple's iphone",
    "Apple is my favorite company",
    "I like apple's ipad",
    "I like apple;s macbook",
]

LangSmith 추적을 시작합니다.
[프로젝트명]
CH10-Retriever


In [2]:
bm25_retriever = BM25Retriever.from_texts(
    doc_list
)
bm25_retriever.k = 1

embedding = OpenAIEmbeddings()
faiss_vectorstore = FAISS.from_texts(
    doc_list,
    embedding,
)
faiss_retriever = faiss_vectorstore.as_retriever(search_kwargs={"k":1})

In [3]:
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.7,0.3],
)

In [4]:
query = "my favorite fruit is apple"
ensemble_result = ensemble_retriever.invoke(query)
bm25_result = bm25_retriever.invoke(query)
faiss_result =faiss_retriever.invoke(query)

print("[Ensemble Retriever]")
for doc in bm25_result:
    print(f"Content: {doc.page_content}")
    print()

print("[BM25 Retriever]")
for doc in bm25_result:
    print(f"Content: {doc.page_content}")
    print()

print("[FAISS Retriever]")
for doc in faiss_result:
    print(f"Content: {doc.page_content}")
    print()

[Ensemble Retriever]
Content: Apple is my favorite company

[BM25 Retriever]
Content: Apple is my favorite company

[FAISS Retriever]
Content: I like apples



In [5]:
from langchain_core.runnables import ConfigurableField

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
).configurable_fields(
    weights=ConfigurableField(
        id="ensemble_weights",
        name="Ensemble Weights",
        description="Ensemble Weights",
    )
)

In [6]:
config = {"configurable": {"ensemble_weights": [1,0]}}

docs = ensemble_retriever.invoke("my favorite fruit is apple", config=config)
docs

[Document(metadata={}, page_content='Apple is my favorite company'),
 Document(id='afed8092-079e-4c43-aacf-f54c8d0cf6f4', metadata={}, page_content='I like apples')]

In [7]:
config = {"configurable": {"ensemble_weights": [0,1]}}

docs = ensemble_retriever.invoke("my favorite furit is apple", config=config)

docs

[Document(id='afed8092-079e-4c43-aacf-f54c8d0cf6f4', metadata={}, page_content='I like apples'),
 Document(metadata={}, page_content='Apple is my favorite company')]